## Pattern {PATTERN_NAME} Index Building Content

This notebook demonstrates how to process documents and build a vector store index for RAG applications. It covers document discovery, text extraction, chunking, and uploading embeddings to a vector database using OpenShift AI Models-as-a-Service (MaaS).

### &#x1F4CB; Contents 
This notebook contains the following sections:

- **[Setup](#Setup)**
  - [Install packages](#Install-packages)
  - [Import required libraries](#Import-required-libraries)
  - [Configure S3 credentials](#Configure-S3-credentials)
  - [Prepare S3 client](#Prepare-S3-client)
- **[Process input documents](#Process-input-documents)**
  - [Documents discovery](#Documents-discovery)
  - [Text extraction](#Text-extraction)
- **[Upload documents content into vector store](#Upload-documents-content-into-vector-store)**
  - [Prepare MaaS client](#Prepare-MaaS-client)
  - [Prepare chunker](#Prepare-chunker)
  - [Initialize vector store](#Initialize-vector-store)
  - [Upload chunks to vector store](#Upload-chunks-to-vector-store)
  - [Retrieve chunks for sample question](#Retrieve-chunks-for-sample-question)
- **[Summary](#Summary)**

---

## Setup

This section sets up the notebook environment by installing required packages, importing libraries, and configuring access to S3 storage.

### Install packages

Install all required Python packages for document processing and RAG operations:
- **ai4rag**: The AutoRAG framework for building RAG applications

In [ ]:
%pip install 'ai4rag[text-extraction]~={AI4RAG_VERSION}' | tail -n 1

1## Prerequisites for Disconnected Clusters

If you are running this notebook on a **disconnected cluster** (no internet access), you must pre-download the required ML models before execution.

**Why:** This notebook uses **Docling** for document processing and **HuggingFace models** for embeddings. By default, these are downloaded on-the-fly from the internet. On disconnected clusters, you must prepare them offline.

**What you need:**
- **Docling artifacts** (core + RapidOCR models) — ~300-400 MB
- **HuggingFace models** (if using custom embeddings) — size varies

**Environment variables:**
- `DOCLING_ARTIFACTS_PATH`: Path to pre-downloaded docling artifacts
- `HF_HOME`: Directory containing pre-downloaded HuggingFace models
- `HF_HUB_OFFLINE`: Set to `"1"` to enforce offline mode

👉 **Skip this section if you have internet access.** Jump to [Import required libraries](#Import-required-libraries).

👉 **For setup instructions, see [Appendix: Downloading Models for Offline Use](#Appendix-Downloading-Models-for-Offline-Use) at the end of this notebook.**

### Configure Models for Disconnected Environments

For disconnected clusters, configure the paths to your pre-downloaded model artifacts before proceeding.

⚠️ **Required for disconnected clusters** | ✅ Skip if you have internet access

In [ ]:
import os
from pathlib import Path

# For disconnected clusters: set these paths to your pre-downloaded artifacts
docling_path = os.getenv("DOCLING_ARTIFACTS_PATH")
hf_home = os.getenv("HF_HOME")

if docling_path:
    print(f"✓ DOCLING_ARTIFACTS_PATH is set: {docling_path}")
    os.environ["HF_HUB_OFFLINE"] = "1"  # Enforce offline mode
    print("✓ Offline mode enabled (HF_HUB_OFFLINE=1)")
else:
    print("ℹ️  DOCLING_ARTIFACTS_PATH not set. Models will be downloaded on-demand.")
    print("   For disconnected clusters, see the appendix for setup instructions.")

if hf_home:
    print(f"✓ HF_HOME is set: {hf_home}")
else:
    print("ℹ️  HF_HOME not set. Will use default HuggingFace cache.")

### Validate Offline Configuration

If using a disconnected cluster, verify that model artifacts exist before starting extraction:

In [ ]:
# Validate artifacts (disconnected clusters only)
if docling_path:
    artifacts_dir = Path(docling_path)
    if not artifacts_dir.exists():
        raise ValueError(f"DOCLING_ARTIFACTS_PATH does not exist: {docling_path}")
    
    print(f"✓ Docling artifacts directory exists: {artifacts_dir}")
    
    # Check for RapidOCR models (optional)
    rapidocr_path = artifacts_dir / "RapidOcr" / "onnx" / "PP-OCRv4"
    if rapidocr_path.exists():
        det_models = list((rapidocr_path / "det").glob("*.onnx"))
        rec_models = list((rapidocr_path / "rec").glob("*.onnx"))
        print(f"✓ RapidOCR models found: {len(det_models)} detection, {len(rec_models)} recognition")
    else:
        print("ℹ️  RapidOCR models not found (OCR will be disabled if no internet)")
else:
    print("✓ Online mode: models will be downloaded as needed")

### Import required libraries

Import all necessary Python modules and configure logging to suppress verbose output from component loggers.

In [ ]:
import getpass
import json
import logging
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

for logger_name in (
    "httpx",
    "documents-discovery",
    "text-extraction",
):
    logging.getLogger(logger_name).propagate = False

### Configure S3 credentials

To load documents from S3-compatible object storage, you need to provide credentials. If you're using OpenShift AI, these can be configured as data connections.

&#x1F4CC; **Action**: Provide the credentials for your S3 instance if they are not already set in the notebook environment.

&#x1F4A1; **Tip**: In the project, open **Connections** and add an **S3 compatible object storage connection** to a bucket you will use for documents and test data. Open **Workbenches**, edit your workbench, and attach the S3 connection you created so the notebook can read from the bucket. Save and restart the workbench if prompted.

In [ ]:
required_vars = ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_S3_ENDPOINT", "AWS_DEFAULT_REGION", "AWS_S3_BUCKET"]
missing = [var for var in required_vars if not os.environ.get(var)]
if missing:
    raise ValueError(f"Missing required environment variables: {{missing}}")

### Prepare S3 client

Creates an S3 client session using the provided credentials. This client will be used to discover and download documents from the specified S3 bucket.

In [ ]:
from ai4rag.utils.clients.s3 import create_s3_client

try:
    s3_client = create_s3_client()
    s3_client.head_bucket(Bucket=os.environ["AWS_S3_BUCKET"])
except Exception as exc:
    from botocore.exceptions import SSLError

    if isinstance(exc, SSLError):
        warnings.warn("SSL verification failed for S3 — retrying with verify=False.")
        s3_client = create_s3_client(verify=False)
    else:
        raise

---

## Process input documents

This section handles document discovery and text extraction. Documents are first discovered in S3 storage, then their content is extracted and saved as DoclingDocument JSON files for further processing.

The data processing pipeline prepares documents for the RAG system in multiple steps. Each step produces outputs stored under `step_outputs/`. 

| Step | Function | Purpose |
|------|----------|---------| 
| 1 | **`discover_documents`** | List documents under every selected prefix, prioritize benchmark-referenced docs, apply a size cap, and write a JSON manifest (no content download). |
| 2 | **`extract_text`** | Download the listed documents from S3 and extract text to DoclingDocument JSON files using Docling. |

In [ ]:
from ai4rag.utils.data.documents_discovery import discover_documents
from ai4rag.utils.data.text_extraction import extract_text

step_output_dir = Path("./step_outputs")
input_data_bucket_name = os.environ["AWS_S3_BUCKET"]
input_data_keys = {INPUT_DATA_KEYS}
step_output_dir.mkdir(parents=True, exist_ok=True)

### Documents discovery

Lists objects in the S3 input bucket, filters by supported extensions (e.g., `.pdf`, `.docx`, `.pptx`, `.md`, `.html`, `.txt`), and builds a document set. Documents referenced in the benchmark are prioritized, then others are added until a configurable size limit (1 GB by default) is reached. This step does not download document contents but writes a JSON manifest (`documents_descriptor.json`) containing the bucket, prefixes, and list of selected object keys and sizes for the next step.

In [ ]:
result = discover_documents(
    bucket_name=input_data_bucket_name,
    prefixes=input_data_keys,
    s3_client=s3_client,
)
result.save(step_output_dir / "discovered_documents")

print(json.dumps(result.to_dict(), indent=4, ensure_ascii=False))

### Text extraction

Reads the `documents_descriptor.json` produced by the discovery step, downloads each listed document from S3 into a temporary directory, and runs **Docling** to extract text. Output is one DoclingDocument JSON file per document (e.g., `report.pdf.json`) written to the artifact output path. Each document's `name` field preserves the original filename including its suffix. These files form the final text corpus for the RAG system.

In [ ]:
extracted_text_dir = step_output_dir / "extracted_text"extraction_result = extract_text(    documents=result.to_dict()["documents"],    bucket=result.bucket,    output_dir=extracted_text_dir,    docling_artifacts_path=os.getenv("DOCLING_ARTIFACTS_PATH"),
)print(    f"Extracted {{extraction_result.processed_count}}/{{extraction_result.total_documents}} documents "    f"({{extraction_result.error_count}} errors)")

---

## Upload documents content into vector store

This section configures the vector store, chunks the extracted documents, and uploads embeddings to the database for semantic search.

### Prepare MaaS client

OpenShift AI Models-as-a-Service (MaaS) serves every model from a single OpenAI-compatible endpoint. This section builds one client from the MaaS base URL and API key; the embedding model reuses it.

**Prerequisites:**
- `MAAS_API_KEY`: Your authentication key for the MaaS API
- `MAAS_BASE_URL`: The complete OpenAI-compatible endpoint URL, used verbatim (e.g. `https://<host>/v1`)

&#x1F4A1; **Tip**: In OpenShift AI Workbench, you can add these as environment variables or data connections to avoid entering them manually each time.

In [ ]:
from ai4rag.utils.clients.maas_client import create_maas_client

MAAS_API_KEY = os.getenv("MAAS_API_KEY") or getpass.getpass("Please enter 'MAAS_API_KEY': ")
MAAS_BASE_URL = os.getenv("MAAS_BASE_URL") or getpass.getpass("Please enter 'MAAS_BASE_URL': ")

client = create_maas_client(base_url=MAAS_BASE_URL, api_key=MAAS_API_KEY)

### Prepare chunker

The chunker splits extracted documents into smaller chunks for more effective retrieval. Configuration includes:
- **Chunking Method**: The algorithm used to split text (e.g., recursive character splitting)
- **Chunk Size**: Maximum number of characters per chunk
- **Chunk Overlap**: Number of overlapping characters between consecutive chunks to preserve context

Proper chunking ensures that retrieved context is both relevant and fits within the model's context window.

In [ ]:
from ai4rag.rag.chunking import LangChainChunker, DoclingChunker

chunking_method = "{CHUNKING_METHOD}"
chunk_size = {CHUNK_SIZE}
chunk_overlap = {CHUNK_OVERLAP}

if chunking_method == "hybrid":
    chunker = DoclingChunker(max_tokens=chunk_size)
else:
    chunker = LangChainChunker(method=chunking_method, chunk_size=chunk_size, chunk_overlap=chunk_overlap)

### Initialize vector store

The vector store holds document embeddings and enables semantic search. The backend is selected from the pattern's provider and configured entirely from environment variables, so no connection details or secrets are stored in this notebook.

**Provider:** `{PROVIDER_TYPE}`

&#x1F4CC; **Action**: Set the environment variables below for the selected provider before running the next cell.

{REQUIRED_ENV_VARS}

&#x1F4A1; **Tip**: In OpenShift AI Workbench, add these as environment variables or data connections so you don't have to set them manually each session.

In [ ]:
from ai4rag.rag.embedding.openai_model import OpenAIEmbeddingModel, OpenAIEmbeddingParams
from ai4rag.rag.vector_store import get_vector_store, get_vector_store_config

embedding_model_id = "{EMBEDDING_MODEL_ID}"
params = OpenAIEmbeddingParams(**{EMBEDDING_PARAMS})

embedding_model = OpenAIEmbeddingModel(client=client, model_id=embedding_model_id, params=params)

provider_type = "{PROVIDER_TYPE}"
collection_name = "{COLLECTION_NAME}"

vector_store_config = get_vector_store_config(provider_type)
vector_store = get_vector_store(
    embedding_model=embedding_model,
    config=vector_store_config,
    collection_name=collection_name,
)

### Upload chunks to vector store

This section processes each extracted DoclingDocument JSON file by:
- Loading the DoclingDocument from its JSON representation
- Splitting it into chunks using the configured chunker
- Generating embeddings and uploading them to the vector store

Once complete, all document chunks are indexed and ready for semantic search queries.

In [ ]:
from ai4rag.utils.docling_io import load_docling_documents

documents = load_docling_documents(extracted_text_dir)

for document in documents:
    chunked_documents = chunker.split_documents([document])
    vector_store.add_documents(chunked_documents)

### Retrieve chunks for sample question

This section demonstrates how to perform a semantic search query against the populated vector store. You can test retrieval by searching for relevant chunks based on a sample question.

In [ ]:
from dataclasses import asdict
from pprint import pprint

sample_question = input()

results = vector_store.search(query=sample_question, k=5)
for chunk in results:
    pprint(asdict(chunk), indent=4)

---

## Summary

This notebook successfully processed documents from S3 storage, extracted their text content using Docling, chunked the text into manageable pieces, and uploaded the embeddings to a vector store. The indexed documents are now ready for semantic search and retrieval in RAG applications.

---

## Appendix: Downloading Models for Offline Use

This section guides you through pre-downloading and transferring ML models to a disconnected cluster.

### Step 1: Download on an Internet-Connected Machine

On a machine **with internet access**, install ai4rag and download artifacts:

```bash
# Install ai4rag with text-extraction support
pip install 'ai4rag[text-extraction]'
```

### Step 2: Download Docling Artifacts

Create a Python script to trigger Docling model downloads:

```python
from pathlib import Path
from docling.document_converter import DocumentConverter
import shutil
import tempfile

# Create a temporary test file to trigger downloads
with tempfile.NamedTemporaryFile(suffix=".txt", delete=False, mode="w") as f:
    f.write("Test document for model download")
    test_file = f.name

try:
    # This will download all docling artifacts to ~/.cache/docling/
    converter = DocumentConverter()
    converter.convert_document_string(test_file)
    print("✓ Docling artifacts downloaded to ~/.cache/docling/")
finally:
    Path(test_file).unlink()

# Copy artifacts to your destination
import os
from_path = Path.home() / ".cache" / "docling"
to_path = Path("/tmp/docling_artifacts")  # Change to your preferred location
to_path.mkdir(parents=True, exist_ok=True)

if from_path.exists():
    shutil.copytree(from_path, to_path / "artifacts", dirs_exist_ok=True)
    print(f"✓ Copied to {to_path / 'artifacts'}")
```

### Step 3: Download HuggingFace Models (If Needed)

If using custom embedding models, pre-download them:

```bash
export HF_HOME=/tmp/huggingface_cache

# Download the model used in this notebook
python -c "from transformers import AutoTokenizer, AutoModel; \
  tokenizer = AutoTokenizer.from_pretrained('BAAI/bge-m3'); \
  model = AutoModel.from_pretrained('BAAI/bge-m3')"

# Or download any other model you're using
python -c "from transformers import AutoTokenizer; \
  AutoTokenizer.from_pretrained('your-model-name')"
```

### Step 4: Transfer Artifacts to the Disconnected Cluster

Copy the downloaded artifacts to your cluster:

```bash
# Using rsync (recommended for large directories)
rsync -av /tmp/docling_artifacts/ user@cluster:/offline/models/docling/
rsync -av /tmp/huggingface_cache/ user@cluster:/offline/models/hf_cache/

# Or using scp for smaller transfers
scp -r /tmp/docling_artifacts user@cluster:/offline/models/
scp -r /tmp/huggingface_cache user@cluster:/offline/models/
```

### Step 5: Set Environment Variables on the Cluster

On your **disconnected cluster**, set the environment variables before running the notebook:

```bash
export DOCLING_ARTIFACTS_PATH=/offline/models/docling/artifacts
export HF_HOME=/offline/models/hf_cache
export HF_HUB_OFFLINE=1  # Enforce offline mode
```

Or add these to your notebook environment (e.g., in OpenShift AI workbench settings).

### Step 6: Verify and Run

Run the notebook with the environment variables set. The validation cells will confirm that artifacts are accessible.